# RAG 4일차

## 평가(Evaluation)!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">비즈니스에서 RAG를 어떻게 평가할지 항상 염두에 두세요</h2>
            <span style="color:#181;">이것은 정확하고 신뢰할 수 있는 RAG 파이프라인을 구축하는 데 매우 중요한 부분입니다. LLM으로 비즈니스 문제를 해결하는 많은 측면에 적용될 수 있습니다. 사람들은 종종 RAG 아키텍처와 프레임워크에만 집중하지만, 더 중요한 것은 바로 평가(evaluation)입니다!</span>
        </td>
    </tr>
</table>

> 💡 **전문가 조언**: RAG 시스템을 프로덕션에 배포하기 전에 체계적인 평가는 필수입니다. MRR(평균 역순위), NDCG(정규화된 누적 할인 이득), 키워드 커버리지 같은 지표를 통해 검색 품질을, 정확도/완전성/관련성으로 응답 품질을 측정합니다.

In [1]:
# evaluation 패키지에서 테스트 모듈 임포트
# evaluation/test.py: 테스트 케이스 로딩 담당
# evaluation/eval.py: 검색 및 답변 품질 평가 담당
from evaluation import test

In [2]:
# tests.jsonl 파일에서 150개의 테스트 케이스를 로드합니다
tests = test.load_tests()

In [3]:
len(tests)

150

In [4]:
# 첫 번째 테스트 케이스 내용 확인
# question: 테스트 질문 | category: 질문 유형 | reference_answer: 정답 | keywords: 핵심 키워드
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [5]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [6]:
# 평가 함수 임포트
# evaluate_retrieval: 검색 품질 평가 (MRR, NDCG, 키워드 커버리지)
# evaluate_answer: 답변 품질 평가 (정확도, 완전성, 관련성)
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [7]:
# 검색 평가 실행
# MRR(Mean Reciprocal Rank): 관련 문서가 몇 번째에 등장하는지의 역수 평균 (높을수록 좋음)
# NDCG: 순위를 고려한 검색 품질 지표 (1.0에 가까울수록 완벽)
# keyword_coverage: 정답 키워드가 검색된 문서에 몇 % 포함되는지
evaluate_retrieval(example)

RetrievalEval(mrr=0.16666666666666666, ndcg=0.28711770538226206, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [11]:
# 답변 품질 평가 실행
# LLM이 생성한 답변을 reference_answer와 비교하여 점수 산출
# 반환값: eval(평가 결과), answer(생성된 답변), chunks(검색된 문서 청크)
eval, answer, chunks = evaluate_answer(example)

In [12]:
eval

AnswerEval(feedback="The answer correctly identifies Maxine as the winner and mentions the IIOTY award in 2023, but it omits Thompson's full name, which is present in the reference. This affects completeness. The relevance is high, as it directly addresses the question about the award winner.", accuracy=5.0, completeness=4.0, relevance=5.0)

In [13]:
# 평가 세부 결과 출력
# accuracy(정확도): 사실 오류 없이 정확한 정보를 담고 있는가 (0~5)
# completeness(완전성): 질문에 필요한 모든 정보를 포함하는가 (0~5)
# relevance(관련성): 질문에 직접적으로 관련된 내용인가 (0~5)
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The answer correctly identifies Maxine as the winner and mentions the IIOTY award in 2023, but it omits Thompson's full name, which is present in the reference. This affects completeness. The relevance is high, as it directly addresses the question about the award winner.
5.0
4.0
5.0
